# MiniMax H3 數字人｜Colab 版（一張照片＋一段提示詞 → 會說話的影片）

由上往下依序執行。**執行階段請選 L4 GPU**（A100 也可以但很耗點數；T4 跑不動）。

| 步驟 | 做什麼 | 時間（主人 L4 實測） |
|---|---|---|
| ① 檢查＋安裝 | GPU／驅動不符就停下，不安裝；通過才裝 ComfyUI＋節點 | 約 1 分鐘 |
| ② 啟動 | 背景啟動 ComfyUI，工作流放進清單 | 約 1 分鐘 |
| ③ 開介面 | Colab 內建代理（推薦）；③-b 是 cloudflare 備援 | 數秒 |
| ④ 下載模型 | 約 37 GB（ComfyUI 確定正常才下載） | 約 1 分鐘 |
| ⑤ 上傳照片 | 人物照一張 | — |
| ⑥ 貼提示詞 | 用 H3 提示詞 skill 產生後整段貼上 | — |
| ⑦ 生成 | 10 秒影片 | 約 4 分鐘 |

只想用介面操作節點：跑 ①②③④ 就好。只想用表單生成影片：③ 可以跳過。

**Hugging Face 權杖（建議）**：左側欄「🔑 密鑰」新增 `HF_TOKEN`（請用**唯讀**權杖），並開啟「筆記本存取權」。

**交給桌面版 AI 生成**：跑 ①②③-b④，把 ③-b 的網址交給裝了 `h3-colab-digital-human` skill 的 AI（skill 在 repo 的 `skills/`）。

模型：`Minimax-h3_Singularity_x_10Eros-Max_PRISM`（int8，加速已融合，6 步）＋官方 Qwen3-VL 文字編碼器與 VAE。授權依照各來源模型。

最新版：<https://github.com/dseditor/minimax-h3-colab>

In [ ]:
#@title ① 檢查環境＋安裝 ComfyUI（GPU 或驅動不符就停下，不會安裝）
FIX_SRC = "# Colab 內建代理會把網址裡的 %2F 解成 /，ComfyUI 的 /userdata/{file} 就配不到 → 工作流清單點開 404。\n# 這個小節點補一條「被拆成兩段」的路由，讀寫同一個使用者資料夾。\nimport os\nfrom aiohttp import web\nimport folder_paths\nfrom server import PromptServer\n\nroutes = PromptServer.instance.routes\nBASE = os.path.abspath(os.path.join(folder_paths.get_user_directory(), 'default'))\n\n\ndef _path(req):\n    p = os.path.abspath(os.path.join(BASE, req.match_info['sub'], req.match_info['file']))\n    if not p.startswith(BASE + os.sep):\n        raise web.HTTPForbidden()\n    return p\n\n\n@routes.get('/userdata/{sub}/{file}')\nasync def _get(req):\n    p = _path(req)\n    if not os.path.isfile(p):\n        return web.Response(status=404)\n    return web.FileResponse(p)\n\n\n@routes.post('/userdata/{sub}/{file}')\nasync def _post(req):\n    p = _path(req)\n    if os.path.exists(p) and req.query.get('overwrite', 'true') == 'false':\n        return web.Response(status=409, text='File already exists')\n    os.makedirs(os.path.dirname(p), exist_ok=True)\n    with open(p, 'wb') as fh:\n        fh.write(await req.read())\n    rel = os.path.relpath(p, BASE).replace(os.sep, '/')\n    if req.query.get('full_info', 'false') == 'true':\n        st = os.stat(p)\n        return web.json_response({'path': rel, 'size': st.st_size, 'modified': st.st_mtime})\n    return web.json_response(rel)\n\n\n@routes.delete('/userdata/{sub}/{file}')\nasync def _delete(req):\n    p = _path(req)\n    if not os.path.isfile(p):\n        return web.Response(status=404)\n    os.remove(p)\n    return web.Response(status=204)\n\n\nNODE_CLASS_MAPPINGS = {}\n"
import torch, shutil, subprocess, sys, os
from pathlib import Path
if not torch.cuda.is_available():
    raise RuntimeError('沒有 GPU：請到「執行階段 → 變更執行階段類型」選 L4 GPU，再重跑這格')
cap = torch.cuda.get_device_capability(0)
drv = subprocess.run(['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
print('GPU:', torch.cuda.get_device_name(0), '| capability', cap, '| VRAM', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), 'GiB')
print('驅動:', drv, '| 目前 PyTorch CUDA:', torch.version.cuda, '| 可用磁碟', round(shutil.disk_usage('/content').free / 2**30, 1), 'GiB')
if cap < (8, 0):
    raise RuntimeError(f'這張卡（capability {cap}）跑不動 int8 模型，沒有安裝任何東西。請換 L4（或 A100）再重跑。')
if int(drv.split('.')[0]) < 580:
    raise RuntimeError(f'驅動 {drv} 不支援 CUDA 13（需要 580 以上），沒有安裝任何東西。請換一台執行階段（L4）再試。')
print('✅ 環境檢查通過，開始安裝')
# HF 權杖先放進環境變數：之後啟動的 ComfyUI（含 ListHelper 的模型下載節點）都會沿用
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN'); print('已讀到 HF_TOKEN')
except Exception:
    print('沒有設定 HF_TOKEN，之後改用匿名下載（人多時可能被限速）')
COMFY = Path('/content/ComfyUI')
if not COMFY.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Comfy-Org/ComfyUI.git', str(COMFY)], check=True)
if not (torch.version.cuda or '').startswith('13'):
    print('安裝 CUDA 13 版 PyTorch（int8 ConvRot 權重需要）…')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'torchvision', 'torchaudio',
                    '--index-url', 'https://download.pytorch.org/whl/cu130'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(COMFY / 'requirements.txt'), 'huggingface_hub', 'hf_xet'], check=True)
# 預裝 ListHelper 節點組（含 ModelDownloader：在介面裡從 Hugging Face 抓模型）
LH = COMFY / 'custom_nodes' / 'ComfyUI-ListHelper'
if not LH.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/dseditor/ComfyUI-ListHelper.git', str(LH)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(LH / 'requirements.txt')], check=True)
# Colab 內建代理（③）會把網址的 %2F 解開，工作流清單點開會 404 → 裝一個補路由的小節點
FIX = COMFY / 'custom_nodes' / 'colab_userdata_slash_fix'
FIX.mkdir(parents=True, exist_ok=True)
(FIX / '__init__.py').write_text(FIX_SRC, encoding='utf-8')
# 節點管理器（ComfyUI-Manager 已併入官方，用 pip 套件＋啟動旗標 --enable-manager）
if (COMFY / 'manager_requirements.txt').exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(COMFY / 'manager_requirements.txt')], check=True)
out = subprocess.run([sys.executable, '-c', 'import torch;print(torch.__version__, torch.version.cuda, torch.cuda.is_available())'],
                     capture_output=True, text=True)
print('ComfyUI 會用的 PyTorch：', out.stdout.strip(), out.stderr[-300:])


In [ ]:
#@title ② 啟動 ComfyUI（背景執行）＋把 H3 數字人工作流放進清單
UI_WORKFLOW = '{\n  "id": "3d566500-43d0-4923-99a5-78fb710a8074",\n  "revision": 0,\n  "last_node_id": 61,\n  "last_link_id": 72,\n  "nodes": [\n    {\n      "id": 55,\n      "type": "ConditioningZeroOut",\n      "pos": [\n        120,\n        680\n      ],\n      "size": [\n        160,\n        30\n      ],\n      "flags": {},\n      "order": 11,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "conditioning",\n          "type": "CONDITIONING",\n          "link": 54\n        }\n      ],\n      "outputs": [\n        {\n          "name": "CONDITIONING",\n          "type": "CONDITIONING",\n          "links": [\n            55\n          ]\n        }\n      ],\n      "properties": {\n        "cnr_id": "comfy-core",\n        "ver": "0.34.0",\n        "Node name for S&R": "ConditioningZeroOut",\n        "ue_properties": {\n          "widget_ue_connectable": {},\n          "input_ue_unconnectable": {},\n          "version": "7.8"\n        }\n      }\n    },\n    {\n      "id": 4,\n      "type": "VAELoader",\n      "pos": [\n        100,\n        560\n      ],\n      "size": [\n        270,\n        60\n      ],\n      "flags": {},\n      "order": 0,\n      "mode": 0,\n      "inputs": [],\n      "outputs": [\n        {\n          "name": "VAE",\n          "type": "VAE",\n          "links": [\n            57,\n            59\n          ]\n        }\n      ],\n      "properties": {\n        "cnr_id": "comfy-core",\n        "ver": "0.34.0",\n        "Node name for S&R": "VAELoader",\n        "ue_properties": {\n          "widget_ue_connectable": {},\n          "input_ue_unconnectable": {},\n          "version": "7.8"\n        }\n      },\n      "widgets_values": [\n        "minimax_h3_audio_vae_fp32.safetensors"\n      ],\n      "widgets_values_named": {\n        "vae_name": "minimax_h3_audio_vae_fp32.safetensors"\n      }\n    },\n    {\n      "id": 2,\n      "type": "CLIPLoader",\n      "pos": [\n        100,\n        270\n      ],\n      "size": [\n        270,\n        110\n      ],\n      "flags": {},\n      "order": 1,\n      "mode": 0,\n      "inputs": [],\n      "outputs": [\n        {\n          "name": "CLIP",\n          "type": "CLIP",\n          "links": [\n            24\n          ]\n        }\n      ],\n      "properties": {\n        "cnr_id": "comfy-core",\n        "ver": "0.34.0",\n        "Node name for S&R": "CLIPLoader",\n        "ue_properties": {\n          "widget_ue_connectable": {},\n          "input_ue_unconnectable": {},\n          "version": "7.8"\n        }\n      },\n      "widgets_values": [\n        "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",\n        "minimax",\n        "default"\n      ],\n      "widgets_values_named": {\n        "clip_name": "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",\n        "type": "minimax",\n        "device": "default"\n      }\n    },\n    {\n      "id": 3,\n      "type": "VAELoader",\n      "pos": [\n        100,\n        430\n      ],\n      "size": [\n        270,\n        60\n      ],\n      "flags": {},\n      "order": 2,\n      "mode": 0,\n      "inputs": [],\n      "outputs": [\n        {\n          "name": "VAE",\n          "type": "VAE",\n          "links": [\n            56,\n            58\n          ]\n        }\n      ],\n      "properties": {\n        "cnr_id": "comfy-core",\n        "ver": "0.34.0",\n        "Node name for S&R": "VAELoader",\n        "ue_properties": {\n          "widget_ue_connectable": {},\n          "input_ue_unconnectable": {},\n          "version": "7.8"\n        }\n      },\n      "widgets_values": [\n        "minimax_h3_video_vae_int8_convrot.safetensors"\n      ],\n      "widgets_values_named": {\n        "vae_name": "minimax_h3_video_vae_int8_convrot.safetensors"\n      }\n    },\n    {\n      "id": 1,\n      "type": "UNETLoader",\n      "pos": [\n        100,\n        120\n      ],\n      "size": [\n        270,\n        90\n      ],\n      "flags": {},\n      "order": 3,\n      "mode": 0,\n      "inputs": [],\n      "outputs": [\n        {\n          "name": "MODEL",\n          "type": "MODEL",\n          "links": [\n            66\n          ]\n        }\n      ],\n      "properties": {\n        "cnr_id": "comfy-core",\n        "ver": "0.34.0",\n        "Node name for S&R": "UNETLoader",\n        "ue_properties": {\n          "widget_ue_connectable": {},\n          "input_ue_unconnectable": {},\n          "version": "7.8"\n        }\n      },\n      "widgets_values": [\n        "Minimax-h3_Singularity_x_10Eros-Max_PRISM_int8.safetensors",\n        "default"\n      ],\n      "widgets_values_named": {\n        "unet_name": "Minimax-h3_Singularity_x_10Eros-Max_PRISM_int8.safetensors",\n        "weight_dtype": "default"\n      }\n    },\n    {\n      "id": 16,\n      "type": "SaveVideo",\n      "pos": [\n        1180,\n        220\n      ],\n      "size": [\n        270,\n        450\n      ],\n      "flags": {},\n      "order": 16,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "video",\n          "type": "VIDEO",\n          "link": 45\n        }\n      ],\n      "outputs": [\n        {\n          "name": "video",\n          "type": "VIDEO",\n          "links": null\n        }\n      ],\n      "properties": {\n        "cnr_id": "comfy-core",\n        "ver": "0.34.0",\n        "Node name for S&R": "SaveVideo",\n        "ue_properties": {\n          "widget_ue_connectable": {},\n          "input_ue_unconnectable": {},\n          "version": "7.8"\n        }\n      },\n      "widgets_values": [\n        "video/xxt_variety_s1",\n        "auto",\n        "auto",\n        "auto"\n      ],\n      "widgets_values_named": {\n        "filename_prefix": "video/xxt_variety_s1",\n        "format": "auto",\n        "format.codec": "auto",\n        "codec": "auto"\n      }\n    },\n    {\n      "id": 13,\n      "type": "VAEDecode",\n      "pos": [\n        830,\n        80\n      ],\n      "size": [\n        140,\n        50\n      ],\n      "flags": {},\n      "order": 13,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "samples",\n          "type": "LATENT",\n          "link": 61\n        },\n        {\n          "name": "vae",\n          "type": "VAE",\n          "link": 56\n        }\n      ],\n      "outputs": [\n        {\n          "name": "IMAGE",\n          "type": "IMAGE",\n          "links": [\n            43\n          ]\n        }\n      ],\n      "properties": {\n        "cnr_id": "comfy-core",\n        "ver": "0.34.0",\n        "Node name for S&R": "VAEDecode",\n        "ue_properties": {\n          "widget_ue_connectable": {},\n          "input_ue_unconnectable": {},\n          "version": "7.8"\n        }\n      }\n    },\n    {\n      "id": 14,\n      "type": "VAEDecodeAudio",\n      "pos": [\n        830,\n        180\n      ],\n      "size": [\n        140,\n        50\n      ],\n      "flags": {},\n      "order": 14,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "samples",\n          "type": "LATENT",\n          "link": 62\n        },\n        {\n          "name": "vae",\n          "type": "VAE",\n          "link": 57\n        }\n      ],\n      "outputs": [\n        {\n          "name": "AUDIO",\n          "type": "AUDIO",\n          "links": [\n            44\n          ]\n        }\n      ],\n      "properties": {\n        "cnr_id": "comfy-core",\n        "ver": "0.34.0",\n        "Node name for S&R": "VAEDecodeAudio",\n        "ue_properties": {\n          "widget_ue_connectable": {},\n          "input_ue_unconnectable": {},\n          "version": "7.8"\n        }\n      }\n    },\n    {\n      "id": 15,\n      "type": "CreateVideo",\n      "pos": [\n        810,\n        270\n      ],\n      "size": [\n        270,\n        150\n      ],\n      "flags": {},\n      "order": 15,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "images",\n          "type": "IMAGE",\n          "link": 43\n        },\n        {\n          "name": "audio",\n          "shape": 7,\n          "type": "AUDIO",\n          "link": 44\n        }\n      ],\n      "outputs": [\n        {\n          "name": "VIDEO",\n          "type": "VIDEO",\n          "links": [\n            45\n          ]\n        }\n      ],\n      "properties": {\n        "cnr_id": "comfy-core",\n        "ver": "0.34.0",\n        "Node name for S&R": "CreateVideo",\n        "ue_properties": {\n          "widget_ue_connectable": {},\n          "input_ue_unconnectable": {},\n          "version": "7.8"\n        }\n      },\n      "widgets_values": [\n        24,\n        8,\n        "sRGB",\n        "none"\n      ],\n      "widgets_values_named": {\n        "fps": 24,\n        "bit_depth": 8,\n        "color_space": "sRGB",\n        "codec": "none"\n      }\n    },\n    {\n      "id": 54,\n      "type": "KSampler",\n      "pos": [\n        810,\n        450\n      ],\n      "size": [\n        270,\n        270\n      ],\n      "flags": {},\n      "order": 12,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "model",\n          "type": "MODEL",\n          "link": 67\n        },\n        {\n          "name": "positive",\n          "type": "CONDITIONING",\n          "link": 52\n        },\n        {\n          "name": "negative",\n          "type": "CONDITIONING",\n          "link": 55\n        },\n        {\n          "name": "latent_image",\n          "type": "LATENT",\n          "link": 60\n        }\n      ],\n      "outputs": [\n        {\n          "name": "LATENT",\n          "type": "LATENT",\n          "links": [\n            61,\n            62\n          ]\n        }\n      ],\n      "properties": {\n        "cnr_id": "comfy-core",\n        "ver": "0.34.0",\n        "Node name for S&R": "KSampler",\n        "ue_properties": {\n          "widget_ue_connectable": {},\n          "input_ue_unconnectable": {},\n          "version": "7.8"\n        }\n      },\n      "widgets_values": [\n        1108164926251196,\n        "randomize",\n        6,\n        1,\n        "res_multistep",\n        "simple",\n        1\n      ],\n      "widgets_values_named": {\n        "seed": 1108164926251196,\n        "control_after_generate": "randomize",\n        "steps": 6,\n        "cfg": 1,\n        "sampler_name": "res_multistep",\n        "scheduler": "simple",\n        "denoise": 1\n      }\n    },\n    {\n      "id": 7,\n      "type": "MiniMaxH3ReferenceToVideo",\n      "pos": [\n        400,\n        130\n      ],\n      "size": [\n        370,\n        470\n      ],\n      "flags": {},\n      "order": 10,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "clip",\n          "type": "CLIP",\n          "link": 24\n        },\n        {\n          "name": "vae",\n          "shape": 7,\n          "type": "VAE",\n          "link": 58\n        },\n        {\n          "name": "audio_vae",\n          "shape": 7,\n          "type": "VAE",\n          "link": 59\n        },\n        {\n          "name": "ref_images.ref_image_0",\n          "shape": 7,\n          "type": "IMAGE",\n          "link": 27\n        },\n        {\n          "name": "ref_images.ref_image_1",\n          "shape": 7,\n          "type": "IMAGE",\n          "link": null\n        },\n        {\n          "name": "ref_videos.ref_video_0",\n          "shape": 7,\n          "type": "IMAGE",\n          "link": null\n        },\n        {\n          "name": "ref_video_audios.ref_video_audio_0",\n          "shape": 7,\n          "type": "AUDIO",\n          "link": null\n        },\n        {\n          "name": "ref_video_audios.ref_video_audio_1",\n          "shape": 7,\n          "type": "AUDIO",\n          "link": null\n        },\n        {\n          "name": "ref_audios.ref_audio_0",\n          "shape": 7,\n          "type": "AUDIO",\n          "link": null\n        },\n        {\n          "name": "width",\n          "type": "INT",\n          "widget": {\n            "name": "width"\n          },\n          "link": 71\n        },\n        {\n          "name": "height",\n          "type": "INT",\n          "widget": {\n            "name": "height"\n          },\n          "link": 72\n        },\n        {\n          "name": "length",\n          "type": "INT",\n          "widget": {\n            "name": "length"\n          },\n          "link": 65\n        }\n      ],\n      "outputs": [\n        {\n          "name": "positive",\n          "type": "CONDITIONING",\n          "links": [\n            52,\n            54\n          ]\n        },\n        {\n          "name": "LATENT",\n          "type": "LATENT",\n          "links": [\n            60\n          ]\n        }\n      ],\n      "properties": {\n        "cnr_id": "comfy-core",\n        "ver": "0.34.0",\n        "Node name for S&R": "MiniMaxH3ReferenceToVideo",\n        "ue_properties": {\n          "widget_ue_connectable": {},\n          "input_ue_unconnectable": {},\n          "version": "7.8"\n        }\n      },\n      "widgets_values": [\n        "integrated_multimodal_description:\\nGlobal visual direction:\\nCreative intent: Deliver an intimate, heart-warming moment featuring a gentle feminine character directly communicating warmth and affection.; Look: Cinematic soft-focus portrait visual, realistic skin textures and delicate lighting.; Lighting: Warm golden hour natural daylight streaming through soft curtains.; Palette: Warm beige, soft cream, pastel peach, and glowing amber tones.; Texture: Tactile wool knit texture, soft skin glow, light airy fabric.; Composition: Centered portrait framing with smooth depth of field blurring the background.; Visual motif: Golden sunbeams and gentle direct eye contact.; Continuity: Keep wardrobe, hairstyle, and soft warm lighting tone consistent between shots.\\nCamera and performance grammar:\\nCamera grammar: Slow, subtle forward push transitions into an intimate close-up framing.; Performance rule: Emphasize realistic micro-expressions, delicate lip movements, and authentic sweet smiles.; Editing rhythm: Calm, continuous flow giving sufficient pacing for clean dialogue delivery.\\nTimeline:\\n[Shot 1, 0.0-4.5s]; The young woman sits in a sunlit cozy room near a window. She turns her head gently towards the camera, her posture relaxed and welcoming, bringing a soft smile to her face as she leans slightly forward.; Performance: Warm, gentle eye contact with eyes sparkling subtly; a tender, affectionate smile forms naturally on her lips.; Environment: Warm golden afternoon sunlight streams through translucent window curtains, highlighting delicate dust motes softly floating in the air.; Visual: Soft rim lighting catching the strands of her hair; creamy background bokeh.; Medium eye-level shot, slowly pushing forward towards the subject.; Transitions from a gentle lingering gaze into an intimate close-up posture.; Beat: Shot pushes closer as her smile widens warm and inviting.; Transition: Smooth spatial cut matching her subtle tilt of head closer to the camera.; Sound: Soft cozy ambient acoustic guitar melody fades in quietly.\\n[Shot 2, 4.5-10.0s]; She tilts her head slightly, rests a hand lightly near her cheek with a sweet expression, and speaks clearly with graceful lip movements.; Performance: Her expression is charming, tender, and deeply affectionate, smiling sweetly while effortlessly speaking the line.; Environment: Golden sunbeams cast a warm glow across her cheek and knit sweater.; Visual: Detailed realistic facial micro-expressions, clear lip movement, warm eye catchlights.; Intimate close-up shot, stable with slight organic hand-held warmth.; Completes speaking the phrase and holds a sweet, lingering smile.; Beat: Dialogue delivery occurs smoothly across the middle of the shot.; Transition: Gently lingers on her adorable smile until a slow dissolve fade out.; Sound: Soft, sweet feminine voice speaking clearly in Chinese: \'想我嗎我在等你\'.\\noverall_soundscape:\\nQuiet, cozy sunlit room ambience with gentle indoor acoustics.\\nnon_diegetic_music:\\nSubtle, romantic acoustic piano and guitar notes playing softly in the background.",\n        640,\n        832,\n        243,\n        "match"\n      ],\n      "widgets_values_named": {\n        "prompt": "integrated_multimodal_description:\\nGlobal visual direction:\\nCreative intent: Deliver an intimate, heart-warming moment featuring a gentle feminine character directly communicating warmth and affection.; Look: Cinematic soft-focus portrait visual, realistic skin textures and delicate lighting.; Lighting: Warm golden hour natural daylight streaming through soft curtains.; Palette: Warm beige, soft cream, pastel peach, and glowing amber tones.; Texture: Tactile wool knit texture, soft skin glow, light airy fabric.; Composition: Centered portrait framing with smooth depth of field blurring the background.; Visual motif: Golden sunbeams and gentle direct eye contact.; Continuity: Keep wardrobe, hairstyle, and soft warm lighting tone consistent between shots.\\nCamera and performance grammar:\\nCamera grammar: Slow, subtle forward push transitions into an intimate close-up framing.; Performance rule: Emphasize realistic micro-expressions, delicate lip movements, and authentic sweet smiles.; Editing rhythm: Calm, continuous flow giving sufficient pacing for clean dialogue delivery.\\nTimeline:\\n[Shot 1, 0.0-4.5s]; The young woman sits in a sunlit cozy room near a window. She turns her head gently towards the camera, her posture relaxed and welcoming, bringing a soft smile to her face as she leans slightly forward.; Performance: Warm, gentle eye contact with eyes sparkling subtly; a tender, affectionate smile forms naturally on her lips.; Environment: Warm golden afternoon sunlight streams through translucent window curtains, highlighting delicate dust motes softly floating in the air.; Visual: Soft rim lighting catching the strands of her hair; creamy background bokeh.; Medium eye-level shot, slowly pushing forward towards the subject.; Transitions from a gentle lingering gaze into an intimate close-up posture.; Beat: Shot pushes closer as her smile widens warm and inviting.; Transition: Smooth spatial cut matching her subtle tilt of head closer to the camera.; Sound: Soft cozy ambient acoustic guitar melody fades in quietly.\\n[Shot 2, 4.5-10.0s]; She tilts her head slightly, rests a hand lightly near her cheek with a sweet expression, and speaks clearly with graceful lip movements.; Performance: Her expression is charming, tender, and deeply affectionate, smiling sweetly while effortlessly speaking the line.; Environment: Golden sunbeams cast a warm glow across her cheek and knit sweater.; Visual: Detailed realistic facial micro-expressions, clear lip movement, warm eye catchlights.; Intimate close-up shot, stable with slight organic hand-held warmth.; Completes speaking the phrase and holds a sweet, lingering smile.; Beat: Dialogue delivery occurs smoothly across the middle of the shot.; Transition: Gently lingers on her adorable smile until a slow dissolve fade out.; Sound: Soft, sweet feminine voice speaking clearly in Chinese: \'想我嗎我在等你\'.\\noverall_soundscape:\\nQuiet, cozy sunlit room ambience with gentle indoor acoustics.\\nnon_diegetic_music:\\nSubtle, romantic acoustic piano and guitar notes playing softly in the background.",\n        "width": 640,\n        "height": 832,\n        "length": 243,\n        "ref_image_size": "match"\n      }\n    },\n    {\n      "id": 5,\n      "type": "LoadImage",\n      "pos": [\n        -200,\n        130\n      ],\n      "size": [\n        270,\n        320\n      ],\n      "flags": {},\n      "order": 4,\n      "mode": 0,\n      "inputs": [],\n      "outputs": [\n        {\n          "name": "IMAGE",\n          "type": "IMAGE",\n          "links": [\n            27,\n            69\n          ]\n        },\n        {\n          "name": "MASK",\n          "type": "MASK",\n          "links": null\n        }\n      ],\n      "properties": {\n        "cnr_id": "comfy-core",\n        "ver": "0.34.0",\n        "Node name for S&R": "LoadImage",\n        "ue_properties": {\n          "widget_ue_connectable": {},\n          "input_ue_unconnectable": {},\n          "version": "7.8"\n        }\n      },\n      "widgets_values": [\n        "LINE_NOTE_251123_1.jpg",\n        "image"\n      ],\n      "widgets_values_named": {\n        "image": "LINE_NOTE_251123_1.jpg",\n        "upload": "image"\n      }\n    },\n    {\n      "id": 56,\n      "type": "PrimitiveInt",\n      "pos": [\n        -210,\n        520\n      ],\n      "size": [\n        270,\n        90\n      ],\n      "flags": {},\n      "order": 5,\n      "mode": 0,\n      "inputs": [],\n      "outputs": [\n        {\n          "name": "INT",\n          "type": "INT",\n          "links": [\n            63\n          ]\n        }\n      ],\n      "title": "Seconds",\n      "properties": {\n        "cnr_id": "comfy-core",\n        "ver": "0.31.0",\n        "Node name for S&R": "PrimitiveInt",\n        "ue_properties": {\n          "widget_ue_connectable": {},\n          "input_ue_unconnectable": {},\n          "version": "7.8"\n        }\n      },\n      "widgets_values": [\n        10,\n        "fixed"\n      ],\n      "widgets_values_named": {\n        "value": 10,\n        "fixed": "fixed"\n      },\n      "color": "#232",\n      "bgcolor": "#353"\n    },\n    {\n      "id": 58,\n      "type": "ComfyMathExpression",\n      "pos": [\n        210,\n        760\n      ],\n      "size": [\n        390,\n        140\n      ],\n      "flags": {\n        "collapsed": true\n      },\n      "order": 8,\n      "mode": 0,\n      "inputs": [\n        {\n          "label": "a",\n          "name": "values.a",\n          "type": "FLOAT,INT,BOOLEAN",\n          "link": 63\n        },\n        {\n          "name": "values.b",\n          "shape": 7,\n          "type": "FLOAT,INT,BOOLEAN",\n          "link": null\n        }\n      ],\n      "outputs": [\n        {\n          "name": "FLOAT",\n          "type": "FLOAT",\n          "links": null\n        },\n        {\n          "name": "INT",\n          "type": "INT",\n          "links": [\n            65\n          ]\n        },\n        {\n          "name": "BOOL",\n          "type": "BOOLEAN",\n          "links": null\n        }\n      ],\n      "properties": {\n        "cnr_id": "comfy-core",\n        "ver": "0.31.0",\n        "Node name for S&R": "ComfyMathExpression",\n        "ue_properties": {\n          "widget_ue_connectable": {},\n          "input_ue_unconnectable": {},\n          "version": "7.8"\n        }\n      },\n      "widgets_values": [\n        "max(5, round(a * 24)) + (5 - (max(5, round(a * 24)) % 17)) % 17\\n"\n      ],\n      "widgets_values_named": {\n        "expression": "max(5, round(a * 24)) + (5 - (max(5, round(a * 24)) % 17)) % 17\\n"\n      }\n    },\n    {\n      "id": 59,\n      "type": "BlockSparseAttention",\n      "pos": [\n        390,\n        650\n      ],\n      "size": [\n        270,\n        250\n      ],\n      "flags": {},\n      "order": 6,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "model",\n          "type": "MODEL",\n          "link": 66\n        }\n      ],\n      "outputs": [\n        {\n          "name": "model",\n          "type": "MODEL",\n          "links": [\n            67\n          ]\n        }\n      ],\n      "properties": {\n        "cnr_id": "comfy-core",\n        "ver": "0.42.0",\n        "Node name for S&R": "BlockSparseAttention",\n        "ue_properties": {\n          "widget_ue_connectable": {},\n          "input_ue_unconnectable": {},\n          "version": "7.8"\n        }\n      },\n      "widgets_values": [\n        "sol-attn",\n        1.3,\n        0.2,\n        1,\n        "",\n        12288,\n        256,\n        "exact_kv_and_rows",\n        false\n      ],\n      "widgets_values_named": {\n        "selection": "sol-attn",\n        "selection.tau": 1.3,\n        "start_percent": 0.2,\n        "end_percent": 1,\n        "dense_blocks": "",\n        "min_tokens": 12288,\n        "extra_tokens": 256,\n        "sink_conditioning": "exact_kv_and_rows",\n        "verbose": false\n      }\n    },\n    {\n      "id": 60,\n      "type": "GetImageSize",\n      "pos": [\n        210,\n        820\n      ],\n      "size": [\n        140,\n        124\n      ],\n      "flags": {},\n      "order": 9,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "image",\n          "type": "IMAGE",\n          "link": 70\n        }\n      ],\n      "outputs": [\n        {\n          "name": "width",\n          "type": "INT",\n          "links": [\n            71\n          ]\n        },\n        {\n          "name": "height",\n          "type": "INT",\n          "links": [\n            72\n          ]\n        },\n        {\n          "name": "batch_size",\n          "type": "INT",\n          "links": null\n        }\n      ],\n      "properties": {\n        "cnr_id": "comfy-core",\n        "ver": "0.42.0",\n        "Node name for S&R": "GetImageSize",\n        "ue_properties": {\n          "widget_ue_connectable": {},\n          "input_ue_unconnectable": {},\n          "version": "7.8"\n        }\n      },\n      "widgets_values": [],\n      "widgets_values_named": {}\n    },\n    {\n      "id": 61,\n      "type": "ImageScaleToTotalPixels",\n      "pos": [\n        -210,\n        670\n      ],\n      "size": [\n        270,\n        110\n      ],\n      "flags": {},\n      "order": 7,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "image",\n          "type": "IMAGE",\n          "link": 69\n        }\n      ],\n      "outputs": [\n        {\n          "name": "IMAGE",\n          "type": "IMAGE",\n          "links": [\n            70\n          ]\n        }\n      ],\n      "properties": {\n        "cnr_id": "comfy-core",\n        "ver": "0.42.0",\n        "Node name for S&R": "ImageScaleToTotalPixels",\n        "ue_properties": {\n          "widget_ue_connectable": {},\n          "input_ue_unconnectable": {},\n          "version": "7.8"\n        }\n      },\n      "widgets_values": [\n        "nearest-exact",\n        0.4,\n        32\n      ],\n      "widgets_values_named": {\n        "upscale_method": "nearest-exact",\n        "megapixels": 0.4,\n        "resolution_steps": 32\n      }\n    }\n  ],\n  "links": [\n    [\n      43,\n      13,\n      0,\n      15,\n      0,\n      "IMAGE"\n    ],\n    [\n      44,\n      14,\n      0,\n      15,\n      1,\n      "AUDIO"\n    ],\n    [\n      52,\n      7,\n      0,\n      54,\n      1,\n      "CONDITIONING"\n    ],\n    [\n      54,\n      7,\n      0,\n      55,\n      0,\n      "CONDITIONING"\n    ],\n    [\n      55,\n      55,\n      0,\n      54,\n      2,\n      "CONDITIONING"\n    ],\n    [\n      56,\n      3,\n      0,\n      13,\n      1,\n      "VAE"\n    ],\n    [\n      57,\n      4,\n      0,\n      14,\n      1,\n      "VAE"\n    ],\n    [\n      60,\n      7,\n      1,\n      54,\n      3,\n      "LATENT"\n    ],\n    [\n      61,\n      54,\n      0,\n      13,\n      0,\n      "LATENT"\n    ],\n    [\n      62,\n      54,\n      0,\n      14,\n      0,\n      "LATENT"\n    ],\n    [\n      67,\n      59,\n      0,\n      54,\n      0,\n      "MODEL"\n    ],\n    [\n      69,\n      5,\n      0,\n      61,\n      0,\n      "IMAGE"\n    ],\n    [\n      70,\n      61,\n      0,\n      60,\n      0,\n      "IMAGE"\n    ],\n    [\n      45,\n      15,\n      0,\n      16,\n      0,\n      "VIDEO"\n    ],\n    [\n      63,\n      56,\n      0,\n      58,\n      0,\n      "INT"\n    ],\n    [\n      66,\n      1,\n      0,\n      59,\n      0,\n      "MODEL"\n    ],\n    [\n      24,\n      2,\n      0,\n      7,\n      0,\n      "CLIP"\n    ],\n    [\n      58,\n      3,\n      0,\n      7,\n      1,\n      "VAE"\n    ],\n    [\n      59,\n      4,\n      0,\n      7,\n      2,\n      "VAE"\n    ],\n    [\n      27,\n      5,\n      0,\n      7,\n      3,\n      "IMAGE"\n    ],\n    [\n      71,\n      60,\n      0,\n      7,\n      9,\n      "INT"\n    ],\n    [\n      72,\n      60,\n      1,\n      7,\n      10,\n      "INT"\n    ],\n    [\n      65,\n      58,\n      1,\n      7,\n      11,\n      "INT"\n    ]\n  ],\n  "groups": [],\n  "config": {},\n  "extra": {\n    "ue_links": [],\n    "ds": {\n      "scale": 1.5416511549149052,\n      "offset": [\n        222.7065733956937,\n        -104.30520004129706\n      ]\n    },\n    "links_added_by_ue": [],\n    "frontendVersion": "1.55.18",\n    "VHS_latentpreview": false,\n    "VHS_latentpreviewrate": 0,\n    "VHS_MetadataImage": true,\n    "VHS_KeepIntermediate": true\n  },\n  "extensions": {\n    "widget_idx_map": {\n      "54": {\n        "seed": 0,\n        "sampler_name": 4,\n        "scheduler": 5\n      }\n    },\n    "seed_widgets": {\n      "54": 0\n    }\n  },\n  "version": 0.4\n}'
import subprocess, time, urllib.request, sys
# --enable-cors-header：經 cloudflare 通道開介面時，瀏覽器的 Origin（trycloudflare 網址）
# 跟轉進來的 Host（127.0.0.1）對不上，ComfyUI 預設會回 403「存取遭拒」；開這個旗標才放行。
if 'server' in globals() and server.poll() is None:
    server.terminate(); server.wait(); print('已關閉舊的 ComfyUI，重新啟動')
# --enable-compress-response-body：介面要抓一大包節點清單和前端檔案，壓縮後走通道快很多。
# 節點管理器安全等級：normal- ＝可從清單裝節點，也可用 Git 網址安裝（本機模式下）。
# 這台 Colab 用完即丟；但 HF_TOKEN 在環境變數裡，自訂節點讀得到 → HF 權杖請用「唯讀」的。
cfg = COMFY / 'user' / '__manager' / 'config.ini'
cfg.parent.mkdir(parents=True, exist_ok=True)
if not cfg.exists():
    cfg.write_text('[default]\nsecurity_level = normal-\n', encoding='utf-8')
# 工作流放進清單（③、③-b 都看得到）；檔名用英文，避免網址編碼在代理上出問題
wf_dir = COMFY / 'user' / 'default' / 'workflows'
wf_dir.mkdir(parents=True, exist_ok=True)
(wf_dir / 'H3_digital_human.json').write_text(UI_WORKFLOW, encoding='utf-8')
old_cn = wf_dir / 'H3數字人.json'
if old_cn.exists(): old_cn.unlink()
Path('/content/H3_digital_human.json').write_text(UI_WORKFLOW, encoding='utf-8')
log = open('/content/comfyui.log', 'w')
server = subprocess.Popen([sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', '8188', '--enable-cors-header', '--enable-manager', '--enable-compress-response-body'],
                          cwd=str(COMFY), stdout=log, stderr=subprocess.STDOUT)
for i in range(180):
    try:
        urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=2)
        print('ComfyUI 已啟動'); break
    except Exception:
        time.sleep(2)
else:
    print(open('/content/comfyui.log').read()[-3000:])
    raise RuntimeError('ComfyUI 沒有啟動成功，看上面的紀錄')


In [ ]:
#@title ③ 開啟 ComfyUI 介面（推薦：Colab 內建代理，快、穩、網址不外流）
# 執行後下面會出現一個連結；顯示成 localhost 也沒關係，直接點它就會開新分頁。
# ⚠️ 開分頁的瀏覽器必須登入「跟這本 Colab 同一個 Google 帳號」；帳號不同就改用下一格 ③-b。
from google.colab import output
output.serve_kernel_port_as_window(8188)
# 如果左側工作流清單打不開（404），按下面下載 JSON，直接拖進 ComfyUI 畫面就好
from google.colab import files
files.download('/content/H3_digital_human.json')


In [ ]:
#@title ③-b（備援）用 cloudflare 網址開介面：瀏覽器登入的 Google 帳號和 Colab 不同時用這格
# 會產生一個 trycloudflare 網址，點開就是 ComfyUI；也是交給桌面版 AI 操作時要給的網址。
# ⚠️ 網址沒有密碼，拿到的人都能用你的 GPU —— 不要貼到公開的地方。
import subprocess, re, time, json, urllib.request
if not Path('/content/cloudflared').exists():
    subprocess.run(['wget', '-q', '-O', '/content/cloudflared',
                    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'], check=True)
    subprocess.run(['chmod', '+x', '/content/cloudflared'], check=True)
# 通道的輸出寫進檔案，不要接 PIPE：PIPE 沒人讀會塞滿，通道就卡住，網頁停在載入畫面
if 'tun' in globals() and tun.poll() is None:
    tun.terminate(); tun.wait()
open('/content/cloudflared.log', 'w').close()
tun = subprocess.Popen(['/content/cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8188'],
                       stdout=open('/content/cloudflared.log', 'a'), stderr=subprocess.STDOUT)
url = None
for i in range(60):
    time.sleep(1)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/cloudflared.log').read())
    if m:
        url = m.group(0); break
print('👉 ComfyUI 介面：', url or '（沒拿到網址，看 /content/cloudflared.log）')


In [ ]:
#@title ④ 下載模型（約 37 GB）
# 預設直接從 Hugging Face 下載（最快）。HF 權杖已在 ① 讀入。
# 只有在 Hugging Face 撞到流量上限時，才勾選 USE_DRIVE_CACHE：
#   勾選後會從「我的雲端硬碟/H3_models」複製（比 HF 慢），雲端硬碟沒有的檔案才去 HF 抓並順便存一份。
USE_DRIVE_CACHE = False  #@param {type:"boolean"}
from huggingface_hub import hf_hub_download
import shutil, os
CACHE = None
if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE = Path('/content/drive/MyDrive/H3_models'); CACHE.mkdir(parents=True, exist_ok=True)
files = [
    ('OliviaRossi/Minimax-h3_Singularity-x-10Eros-Max_PRISM', 'Minimax-h3_Singularity_x_10Eros-Max_PRISM_int8.safetensors', 'diffusion_models'),
    ('Comfy-Org/MiniMax-H3', 'text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors', 'text_encoders'),
    ('Comfy-Org/MiniMax-H3', 'vae/minimax_h3_video_vae_int8_convrot.safetensors', 'vae'),
    ('Comfy-Org/MiniMax-H3', 'vae/minimax_h3_audio_vae_fp32.safetensors', 'vae'),
]
for repo, name, folder in files:
    target = COMFY / 'models' / folder / Path(name).name
    target.parent.mkdir(parents=True, exist_ok=True)
    cached = CACHE / Path(name).name if CACHE else None
    if not target.is_file():
        if cached and cached.is_file():
            print('從雲端硬碟複製', cached.name); shutil.copy(str(cached), str(target))
        else:
            got = Path(hf_hub_download(repo_id=repo, filename=name, local_dir='/content/hf'))
            shutil.move(str(got), str(target))
    if cached and not cached.is_file():
        print('存一份到雲端硬碟', cached.name); shutil.copy(str(target), str(cached))
    print(folder, target.name, f'{target.stat().st_size / 2**30:.1f} GiB')


In [ ]:
#@title ⑤ 上傳人物照（一張）
from google.colab import files
from IPython.display import Image as _Img, display
up = files.upload()
IMAGE_NAME = list(up)[0]
(COMFY / 'input').mkdir(exist_ok=True)
(COMFY / 'input' / IMAGE_NAME).write_bytes(up[IMAGE_NAME])
display(_Img(data=up[IMAGE_NAME], width=320))


In [ ]:
#@title ⑥ 貼上提示詞＋設定（執行後下方會出現輸入框）
# 執行這格後，下面會出現一個大的輸入框：把 skill 產生的整段提示詞直接貼進去就好，
# 不用管引號、換行。貼完不用再執行這格，直接執行 ⑦。
import ipywidgets as W
from IPython.display import display
SAMPLE_PROMPT = "integrated_multimodal_description:\nGlobal visual direction:\nCreative intent: Deliver an intimate, heart-warming moment featuring a gentle feminine character directly communicating warmth and affection.; Look: Cinematic soft-focus portrait visual, realistic skin textures and delicate lighting.; Lighting: Warm golden hour natural daylight streaming through soft curtains.; Palette: Warm beige, soft cream, pastel peach, and glowing amber tones.; Texture: Tactile wool knit texture, soft skin glow, light airy fabric.; Composition: Centered portrait framing with smooth depth of field blurring the background.; Visual motif: Golden sunbeams and gentle direct eye contact.; Continuity: Keep wardrobe, hairstyle, and soft warm lighting tone consistent between shots.\nCamera and performance grammar:\nCamera grammar: Slow, subtle forward push transitions into an intimate close-up framing.; Performance rule: Emphasize realistic micro-expressions, delicate lip movements, and authentic sweet smiles.; Editing rhythm: Calm, continuous flow giving sufficient pacing for clean dialogue delivery.\nTimeline:\n[Shot 1, 0.0-4.5s]; The young woman sits in a sunlit cozy room near a window. She turns her head gently towards the camera, her posture relaxed and welcoming, bringing a soft smile to her face as she leans slightly forward.; Performance: Warm, gentle eye contact with eyes sparkling subtly; a tender, affectionate smile forms naturally on her lips.; Environment: Warm golden afternoon sunlight streams through translucent window curtains, highlighting delicate dust motes softly floating in the air.; Visual: Soft rim lighting catching the strands of her hair; creamy background bokeh.; Medium eye-level shot, slowly pushing forward towards the subject.; Transitions from a gentle lingering gaze into an intimate close-up posture.; Beat: Shot pushes closer as her smile widens warm and inviting.; Transition: Smooth spatial cut matching her subtle tilt of head closer to the camera.; Sound: Soft cozy ambient acoustic guitar melody fades in quietly.\n[Shot 2, 4.5-10.0s]; She tilts her head slightly, rests a hand lightly near her cheek with a sweet expression, and speaks clearly with graceful lip movements.; Performance: Her expression is charming, tender, and deeply affectionate, smiling sweetly while effortlessly speaking the line.; Environment: Golden sunbeams cast a warm glow across her cheek and knit sweater.; Visual: Detailed realistic facial micro-expressions, clear lip movement, warm eye catchlights.; Intimate close-up shot, stable with slight organic hand-held warmth.; Completes speaking the phrase and holds a sweet, lingering smile.; Beat: Dialogue delivery occurs smoothly across the middle of the shot.; Transition: Gently lingers on her adorable smile until a slow dissolve fade out.; Sound: Soft, sweet feminine voice speaking clearly in Chinese: '想我嗎我在等你'.\noverall_soundscape:\nQuiet, cozy sunlit room ambience with gentle indoor acoustics.\nnon_diegetic_music:\nSubtle, romantic acoustic piano and guitar notes playing softly in the background."
_keep = prompt_box.value if 'prompt_box' in globals() else SAMPLE_PROMPT   # 重跑這格不會洗掉已貼的內容
prompt_box = W.Textarea(value=_keep, placeholder='把整段提示詞貼在這裡',
                        layout=W.Layout(width='100%', height='320px'))
dur_box = W.IntSlider(value=10, min=5, max=15, step=1, description='秒數')
mp_box = W.FloatText(value=0.4, description='百萬像素')
seed_box = W.IntText(value=-1, description='seed（-1 隨機）')
clear_btn = W.Button(description='清空輸入框')
clear_btn.on_click(lambda _: setattr(prompt_box, 'value', ''))
count = W.HTML()
def _count(change=None):
    n = len(prompt_box.value.split())
    count.value = f'<b>{n}</b> 個英文字（官方建議 350～500）'
prompt_box.observe(_count, names='value'); _count()
display(W.VBox([W.HBox([clear_btn, count]), prompt_box, dur_box, mp_box, seed_box]))


In [ ]:
#@title ⑦ 生成影片
import json, random, time, urllib.request
from IPython.display import Video, display
WORKFLOW = json.loads('{"55": {"class_type": "ConditioningZeroOut", "inputs": {"conditioning": ["7", 0]}, "_meta": {"title": "ConditioningZeroOut"}}, "4": {"class_type": "VAELoader", "inputs": {"vae_name": "minimax_h3_audio_vae_fp32.safetensors"}, "_meta": {"title": "VAELoader"}}, "2": {"class_type": "CLIPLoader", "inputs": {"clip_name": "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors", "type": "minimax", "device": "default"}, "_meta": {"title": "CLIPLoader"}}, "3": {"class_type": "VAELoader", "inputs": {"vae_name": "minimax_h3_video_vae_int8_convrot.safetensors"}, "_meta": {"title": "VAELoader"}}, "1": {"class_type": "UNETLoader", "inputs": {"unet_name": "Minimax-h3_Singularity_x_10Eros-Max_PRISM_int8.safetensors", "weight_dtype": "default"}, "_meta": {"title": "UNETLoader"}}, "16": {"class_type": "SaveVideo", "inputs": {"filename_prefix": "h3_colab/xxt", "format": "auto", "format.codec": "auto", "codec": "auto", "video": ["15", 0]}, "_meta": {"title": "SaveVideo"}}, "13": {"class_type": "VAEDecode", "inputs": {"samples": ["54", 0], "vae": ["3", 0]}, "_meta": {"title": "VAEDecode"}}, "14": {"class_type": "VAEDecodeAudio", "inputs": {"samples": ["54", 0], "vae": ["4", 0]}, "_meta": {"title": "VAEDecodeAudio"}}, "15": {"class_type": "CreateVideo", "inputs": {"fps": 24, "bit_depth": 8, "color_space": "sRGB", "codec": "none", "images": ["13", 0], "audio": ["14", 0]}, "_meta": {"title": "CreateVideo"}}, "54": {"class_type": "KSampler", "inputs": {"seed": 1108164926251196, "steps": 6, "cfg": 1, "sampler_name": "res_multistep", "scheduler": "simple", "denoise": 1, "model": ["59", 0], "positive": ["7", 0], "negative": ["55", 0], "latent_image": ["7", 1]}, "_meta": {"title": "KSampler"}}, "7": {"class_type": "MiniMaxH3ReferenceToVideo", "inputs": {"prompt": "", "width": ["60", 0], "height": ["60", 1], "length": ["58", 1], "ref_image_size": "match", "clip": ["2", 0], "vae": ["3", 0], "audio_vae": ["4", 0], "ref_images.ref_image_0": ["5", 0]}, "_meta": {"title": "MiniMaxH3ReferenceToVideo"}}, "5": {"class_type": "LoadImage", "inputs": {"image": "LINE_NOTE_251123_1.jpg"}, "_meta": {"title": "LoadImage"}}, "56": {"class_type": "PrimitiveInt", "inputs": {"value": 10}, "_meta": {"title": "Seconds"}}, "58": {"class_type": "ComfyMathExpression", "inputs": {"expression": "max(5, round(a * 24)) + (5 - (max(5, round(a * 24)) % 17)) % 17", "values.a": ["56", 0]}, "_meta": {"title": "ComfyMathExpression"}}, "59": {"class_type": "BlockSparseAttention", "inputs": {"selection": "sol-attn", "selection.tau": 1.3, "start_percent": 0.2, "end_percent": 1, "dense_blocks": "", "min_tokens": 12288, "extra_tokens": 256, "sink_conditioning": "exact_kv_and_rows", "verbose": false, "model": ["1", 0]}, "_meta": {"title": "BlockSparseAttention"}}, "60": {"class_type": "GetImageSize", "inputs": {"image": ["61", 0]}, "_meta": {"title": "GetImageSize"}}, "61": {"class_type": "ImageScaleToTotalPixels", "inputs": {"upscale_method": "nearest-exact", "megapixels": 0.4, "resolution_steps": 32, "image": ["5", 0]}, "_meta": {"title": "ImageScaleToTotalPixels"}}}')
p = json.loads(json.dumps(WORKFLOW))
PROMPT = prompt_box.value.strip()
assert PROMPT, '⑥ 的輸入框是空的，請先貼上提示詞'
print('這次送出的提示詞開頭：', PROMPT[:160].replace(chr(10), ' '), '…（共', len(PROMPT.split()), '字）')
DURATION_SECONDS, MEGAPIXELS, SEED = dur_box.value, mp_box.value, seed_box.value
p['7']['inputs']['prompt'] = PROMPT
p['5']['inputs']['image'] = IMAGE_NAME
p['56']['inputs']['value'] = int(DURATION_SECONDS)
p['61']['inputs']['megapixels'] = float(MEGAPIXELS)
p['54']['inputs']['seed'] = SEED if SEED >= 0 else random.randint(0, 2**50)
req = urllib.request.Request('http://127.0.0.1:8188/prompt', data=json.dumps({'prompt': p}).encode(),
                             headers={'Content-Type': 'application/json'})
try:
    pid = json.loads(urllib.request.urlopen(req).read())['prompt_id']
except urllib.error.HTTPError as e:
    raise RuntimeError(e.read().decode()[:2000])
t0 = time.time(); print('已送出，生成中…（seed', p['54']['inputs']['seed'], '）')
while True:
    time.sleep(10)
    h = json.loads(urllib.request.urlopen('http://127.0.0.1:8188/history/' + pid).read())
    if pid in h:
        st = h[pid]['status']
        if st.get('status_str') == 'error':
            print(json.dumps(st, ensure_ascii=False)[:3000]); raise RuntimeError('生成失敗')
        break
    print(f'  {int(time.time() - t0)} 秒', end='\r')
vids = [v for o in h[pid]['outputs'].values() for v in o.get('images', []) + o.get('videos', []) if v['filename'].endswith('.mp4')]
out = COMFY / 'output' / vids[0]['subfolder'] / vids[0]['filename']
print(f'完成，{(time.time() - t0) / 60:.1f} 分鐘：', out)
display(Video(str(out), embed=True, width=640))
from google.colab import files; files.download(str(out))


In [ ]:
#@title 🔍 出問題時跑這格：看 GPU 與 ComfyUI 紀錄
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
log = open('/content/comfyui.log', encoding='utf-8', errors='ignore').read()
keys = ('error', 'Error', 'int8', 'quant', 'fallback', 'out of memory', 'loaded', 'Prompt executed')
print('\n'.join(l for l in log.splitlines() if any(k in l for k in keys))[-4000:])
